# Painel contábil descritivo de 2019 a 2025

ImpactUFSCar — Área de Pesquisa & Mercado

O painel acompanha as versões que constavam das capturas anuais. A exploração descreve cobertura e consistência contábil. As datas de divulgação ainda estão em diligência. Gere o painel conforme o [README](../README.md).

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
pasta = raiz / Path(os.environ.get("QUANTDISTRESS_PAINEL", "data/processed/sprint02"))
manifesto = json.loads((pasta / "manifesto.json").read_text(encoding="utf-8"))
assert manifesto["modo_temporal"] == "descritivo_revisado"
setores = pd.read_csv(pasta / "cobertura_setor.csv")
indicadores = pd.read_csv(pasta / "cobertura_indicadores.csv")

display(pd.DataFrame(manifesto["anos"])[[
    "ano", "registros_indice_dfp", "observacoes_universo", "fatos_na_fonte", "fatos_analiticos",
    "chaves_contabeis_nao_resolvidas",
]])

## Presença de fatos por exercício, grupo setorial e escopo

Cada ano aparece em dois escopos. A contagem de observações inclui referências sem fatos no escopo. Um emissor pode contribuir em múltiplos exercícios.

In [ ]:
por_grupo = setores.groupby(
    ["ANO_ARQUIVO", "ESCOPO", "GRUPO_SETORIAL"],
)[["OBSERVACOES", "COM_FATOS", "COM_FCA", "COM_CANCELAMENTO_POSTERIOR"]].sum()
por_grupo["PROPORCAO_COM_FATOS"] = por_grupo["COM_FATOS"] / por_grupo["OBSERVACOES"]
display(por_grupo)

## Cobertura dos protótipos por setor

Altere a seleção de ano, escopo e indicador para revisar denominadores e a primeira causa de indisponibilidade. Cada proporção usa todas as referências DFP no grupo e setor correspondentes.

In [ ]:
ano = 2024
escopo = "con"
indicador = "margem_operacional"
recorte = indicadores.loc[
    indicadores["ANO_ARQUIVO"].eq(ano)
    & indicadores["ESCOPO"].eq(escopo)
    & indicadores["INDICADOR"].eq(indicador)
].copy()
recorte["SETOR_FCA"] = recorte["SETOR_FCA"].fillna("Sem setor associado")
resumo = recorte.pivot_table(
    index=["GRUPO_SETORIAL", "SETOR_FCA"], columns="SITUACAO", values="OBSERVACOES",
    aggfunc="sum", fill_value=0,
)
resumo["TOTAL"] = resumo.sum(axis=1)
resumo["PROPORCAO_VALIDA"] = resumo.get("valido", 0) / resumo["TOTAL"]
display(resumo.sort_values("TOTAL", ascending=False))

## Linhagem de uma observação válida

O exemplo mostra as contas de origem de uma observação do recorte. A identificação física reúne o SHA-256 da captura, o membro CSV e o número do registro.

In [ ]:
fatos_indicadores = pd.read_parquet(pasta / str(ano) / "indicadores.parquet")
amostra = fatos_indicadores.loc[
    fatos_indicadores["ESCOPO"].eq(escopo)
    & fatos_indicadores["INDICADOR"].eq(indicador)
    & fatos_indicadores["SITUACAO"].eq("valido")
].head(1)
if amostra.empty:
    display("Nenhuma observação válida neste recorte.")
else:
    display(amostra[["CD_CVM", "DT_REFER", "VERSAO", "ESCOPO", "VALOR", "SHA256_DFP"]])
    display(pd.DataFrame(json.loads(amostra["FONTES"].iloc[0])))